# Validation and Difficulty Scoring

When creating simulated data for benchmarking, two critical questions arise:

1. **How realistic is the simulation?** (Validation)
2. **How difficult is the classification task?** (Difficulty Scoring)

PointillSim provides tools to answer both questions quantitatively.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

np.random.seed(42)

from pointillsim import (
    TissueCellTypes,
    CellTypesProperties,
    HybISS_Setup,
    FOVDistribution,
    FrameWideElement,
    VacuolatedStructure,
    RandomCellTypeRule,
    MixOfNCellTypesRule,
    DistanceBasedRule,
)

# Import validation module
from pointillsim.validation import (
    DifficultyScorer,
    ValidationMetrics,
)

plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.facecolor'] = 'white'
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Create sample tissues with different characteristics
n_cell_types = 5
n_genes = 50
frame_size = 600

# Easy tissue: Clear markers, high concentration
tissue_easy = TissueCellTypes()
tissue_easy.generate_types_and_markers(
    n_genes=n_genes,
    n_cell_types=n_cell_types,
    expected_level=20.0,      # High expression
    concentration=0.95,        # Very distinct markers
)

# Hard tissue: Overlapping markers, low concentration
tissue_hard = TissueCellTypes()
tissue_hard.generate_types_and_markers(
    n_genes=n_genes,
    n_cell_types=n_cell_types,
    expected_level=5.0,        # Low expression
    concentration=0.5,         # Overlapping markers
)

cell_props = CellTypesProperties(n_cell_types=n_cell_types)

print("Created two tissue profiles:")
print(f"  Easy: concentration={0.95}, expression level={20.0}")
print(f"  Hard: concentration={0.5}, expression level={5.0}")

---
## 1. Difficulty Scoring

The `DifficultyScorer` quantifies how challenging a cell type classification task is. It uses:

- **Separability metrics**: How distinct are the cell type expression profiles?
- **Spatial complexity**: How mixed are cell types spatially?
- **Optional classifier accuracy**: Actual classification performance estimate

In [ ]:
# Create difficulty scorers
scorer_easy = DifficultyScorer(tissue_easy)
scorer_hard = DifficultyScorer(tissue_hard)

print("DifficultyScorer initialized")

In [ ]:
# Generate FOVs for scoring
def generate_fov(tissue):
    fov_dist = FOVDistribution(
        frame_size=frame_size,
        background_element=lambda: FrameWideElement(
            frame_size=frame_size,
            tipical_cell_spacing=15,
            rules=RandomCellTypeRule(n_cell_types=n_cell_types)
        ),
        other_elements=[
            lambda: VacuolatedStructure(
                frame_size=frame_size,
                scale=80,
                hole_scale_factor=0.4,
                tipical_cell_spacing=10,
                rules=DistanceBasedRule(
                    n_cell_types=n_cell_types,
                    inner_type=2,
                    outer_type=3,
                )
            )
        ],
        elements_frequency=[1.0],
        attempts_at_elements=3,
    )
    
    fov = fov_dist.generate_fov()
    cell_props.apply(fov)
    
    hybiss = HybISS_Setup(tissue)
    hybiss.observe_dots(fov)
    
    return fov, hybiss

np.random.seed(42)
fov_easy, hybiss_easy = generate_fov(tissue_easy)
np.random.seed(42)
fov_hard, hybiss_hard = generate_fov(tissue_hard)

print(f"Generated FOVs: {fov_easy.n_cells} cells each")

In [ ]:
# Score both tissues
score_easy = scorer_easy.score(fov_easy)
score_hard = scorer_hard.score(fov_hard)

print("Difficulty Scores:")
print(f"\n  Easy tissue:")
for metric, value in score_easy.items():
    print(f"    {metric}: {value:.3f}")

print(f"\n  Hard tissue:")
for metric, value in score_hard.items():
    print(f"    {metric}: {value:.3f}")

In [ ]:
# Visualize the difficulty difference
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Row 1: Easy tissue
ax = axes[0, 0]
ax.scatter(fov_easy.cell_centroids[:, 0], fov_easy.cell_centroids[:, 1],
           c=fov_easy.class_instance, cmap='Set1', s=20, alpha=0.7)
ax.set_xlim(0, frame_size)
ax.set_ylim(0, frame_size)
ax.set_aspect('equal')
ax.set_title('Easy Tissue: Cell Types')

ax = axes[0, 1]
dots_easy = hybiss_easy.make_pandas_df()
ax.scatter(dots_easy['x'], dots_easy['y'], s=1, alpha=0.3, c='blue')
ax.set_xlim(0, frame_size)
ax.set_ylim(0, frame_size)
ax.set_aspect('equal')
ax.set_title(f'Easy: Transcripts ({len(dots_easy):,} dots)')

ax = axes[0, 2]
im = ax.imshow(np.log1p(tissue_easy.gene_expression_by_type[:20, :]), 
               aspect='auto', cmap='viridis')
ax.set_xlabel('Cell Type')
ax.set_ylabel('Gene')
ax.set_title('Easy: Expression Profile\n(Clear markers)')
plt.colorbar(im, ax=ax, label='log(expr+1)')

# Row 2: Hard tissue
ax = axes[1, 0]
ax.scatter(fov_hard.cell_centroids[:, 0], fov_hard.cell_centroids[:, 1],
           c=fov_hard.class_instance, cmap='Set1', s=20, alpha=0.7)
ax.set_xlim(0, frame_size)
ax.set_ylim(0, frame_size)
ax.set_aspect('equal')
ax.set_title('Hard Tissue: Cell Types')

ax = axes[1, 1]
dots_hard = hybiss_hard.make_pandas_df()
ax.scatter(dots_hard['x'], dots_hard['y'], s=1, alpha=0.3, c='red')
ax.set_xlim(0, frame_size)
ax.set_ylim(0, frame_size)
ax.set_aspect('equal')
ax.set_title(f'Hard: Transcripts ({len(dots_hard):,} dots)')

ax = axes[1, 2]
im = ax.imshow(np.log1p(tissue_hard.gene_expression_by_type[:20, :]), 
               aspect='auto', cmap='viridis')
ax.set_xlabel('Cell Type')
ax.set_ylabel('Gene')
ax.set_title('Hard: Expression Profile\n(Overlapping markers)')
plt.colorbar(im, ax=ax, label='log(expr+1)')

plt.suptitle('Difficulty Comparison: Easy vs Hard Classification', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Compare scores visually
fig, ax = plt.subplots(figsize=(10, 6))

metrics = list(score_easy.keys())
easy_values = [score_easy[m] for m in metrics]
hard_values = [score_hard[m] for m in metrics]

x = np.arange(len(metrics))
width = 0.35

bars1 = ax.bar(x - width/2, easy_values, width, label='Easy', color='green', alpha=0.7)
bars2 = ax.bar(x + width/2, hard_values, width, label='Hard', color='red', alpha=0.7)

ax.set_ylabel('Score')
ax.set_title('Difficulty Metrics Comparison')
ax.set_xticks(x)
ax.set_xticklabels(metrics, rotation=45, ha='right')
ax.legend()

# Add value labels
for bar, val in zip(bars1, easy_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.2f}', ha='center', va='bottom', fontsize=9)
for bar, val in zip(bars2, hard_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  - Higher separability = easier classification")
print("  - Lower spatial_complexity = more organized tissue")
print("  - Higher overall_difficulty = harder task")

---
## 2. Classifier-Based Difficulty Estimation

For a more practical difficulty measure, we can train a classifier and measure its accuracy.

In [ ]:
# Generate transcript count matrices
def get_count_matrix(fov, dots_df, tissue):
    """Convert dots to cell × gene count matrix."""
    n_cells = fov.n_cells
    n_genes = tissue.n_genes
    
    # Initialize count matrix
    counts = np.zeros((n_cells, n_genes))
    
    # Gene name to index mapping
    gene_to_idx = {g: i for i, g in enumerate(tissue.gene_names)}
    
    # Count transcripts per cell per gene
    for _, row in dots_df.iterrows():
        cell_idx = int(row['cell'])
        gene_idx = gene_to_idx.get(row['gene'], -1)
        if 0 <= cell_idx < n_cells and gene_idx >= 0:
            counts[cell_idx, gene_idx] += 1
    
    return counts

counts_easy = get_count_matrix(fov_easy, dots_easy, tissue_easy)
counts_hard = get_count_matrix(fov_hard, dots_hard, tissue_hard)

print(f"Count matrices: {counts_easy.shape}")

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

def estimate_classifier_accuracy(counts, labels, classifier='logistic'):
    """Estimate classification accuracy using cross-validation."""
    # Normalize counts
    scaler = StandardScaler()
    X = scaler.fit_transform(np.log1p(counts))
    y = labels
    
    if classifier == 'logistic':
        clf = LogisticRegression(max_iter=1000, random_state=42)
    else:
        clf = RandomForestClassifier(n_estimators=100, random_state=42)
    
    scores = cross_val_score(clf, X, y, cv=5, scoring='accuracy')
    
    return scores.mean(), scores.std()

# Classify easy tissue
acc_easy_lr, std_easy_lr = estimate_classifier_accuracy(
    counts_easy, fov_easy.class_instance, 'logistic'
)
acc_easy_rf, std_easy_rf = estimate_classifier_accuracy(
    counts_easy, fov_easy.class_instance, 'rf'
)

# Classify hard tissue
acc_hard_lr, std_hard_lr = estimate_classifier_accuracy(
    counts_hard, fov_hard.class_instance, 'logistic'
)
acc_hard_rf, std_hard_rf = estimate_classifier_accuracy(
    counts_hard, fov_hard.class_instance, 'rf'
)

print("Classification Accuracy (5-fold CV):")
print(f"\n  Easy Tissue:")
print(f"    Logistic Regression: {acc_easy_lr:.1%} ± {std_easy_lr:.1%}")
print(f"    Random Forest:       {acc_easy_rf:.1%} ± {std_easy_rf:.1%}")
print(f"\n  Hard Tissue:")
print(f"    Logistic Regression: {acc_hard_lr:.1%} ± {std_hard_lr:.1%}")
print(f"    Random Forest:       {acc_hard_rf:.1%} ± {std_hard_rf:.1%}")

In [ ]:
# Visualize in reduced dimensions
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# PCA on easy tissue
pca = PCA(n_components=2)
X_easy_pca = pca.fit_transform(np.log1p(counts_easy))

ax = axes[0]
scatter = ax.scatter(X_easy_pca[:, 0], X_easy_pca[:, 1],
                     c=fov_easy.class_instance, cmap='Set1',
                     s=30, alpha=0.7)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)')
ax.set_title(f'Easy Tissue: PCA\nAccuracy: {acc_easy_rf:.1%}')
plt.colorbar(scatter, ax=ax, label='Cell Type')

# PCA on hard tissue
X_hard_pca = pca.fit_transform(np.log1p(counts_hard))

ax = axes[1]
scatter = ax.scatter(X_hard_pca[:, 0], X_hard_pca[:, 1],
                     c=fov_hard.class_instance, cmap='Set1',
                     s=30, alpha=0.7)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)')
ax.set_title(f'Hard Tissue: PCA\nAccuracy: {acc_hard_rf:.1%}')
plt.colorbar(scatter, ax=ax, label='Cell Type')

plt.suptitle('PCA Visualization: Cell Type Separability', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 3. Validation Metrics

The `ValidationMetrics` class compares simulated data to real reference data to assess realism.

In [ ]:
# Create a "reference" dataset (simulated with specific parameters)
np.random.seed(123)

# Reference tissue
tissue_ref = TissueCellTypes()
tissue_ref.generate_types_and_markers(
    n_genes=n_genes,
    n_cell_types=n_cell_types,
    expected_level=12.0,
    concentration=0.8,
)

# Generate reference FOV
fov_ref, hybiss_ref = generate_fov(tissue_ref)
dots_ref = hybiss_ref.make_pandas_df()
counts_ref = get_count_matrix(fov_ref, dots_ref, tissue_ref)

print(f"Reference dataset: {fov_ref.n_cells} cells, {len(dots_ref)} dots")

In [ ]:
# Create ValidationMetrics
validator = ValidationMetrics()

# Compare easy tissue to reference
metrics_easy = validator.compare(
    simulated_counts=counts_easy,
    simulated_types=fov_easy.class_instance,
    reference_counts=counts_ref,
    reference_types=fov_ref.class_instance,
)

# Compare hard tissue to reference
metrics_hard = validator.compare(
    simulated_counts=counts_hard,
    simulated_types=fov_hard.class_instance,
    reference_counts=counts_ref,
    reference_types=fov_ref.class_instance,
)

print("Validation Metrics (vs Reference):")
print(f"\n  Easy Tissue:")
for metric, value in metrics_easy.items():
    print(f"    {metric}: {value:.3f}")

print(f"\n  Hard Tissue:")
for metric, value in metrics_hard.items():
    print(f"    {metric}: {value:.3f}")

In [ ]:
# Detailed comparison visualizations
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Row 1: Expression distributions
ax = axes[0, 0]
ax.hist(np.log1p(counts_ref.sum(axis=1)), bins=30, alpha=0.7, label='Reference', color='gray')
ax.hist(np.log1p(counts_easy.sum(axis=1)), bins=30, alpha=0.5, label='Easy', color='green')
ax.set_xlabel('log(Total Counts + 1)')
ax.set_ylabel('Frequency')
ax.set_title('Total Counts per Cell')
ax.legend()

ax = axes[0, 1]
ax.hist(np.log1p(counts_ref.sum(axis=1)), bins=30, alpha=0.7, label='Reference', color='gray')
ax.hist(np.log1p(counts_hard.sum(axis=1)), bins=30, alpha=0.5, label='Hard', color='red')
ax.set_xlabel('log(Total Counts + 1)')
ax.set_ylabel('Frequency')
ax.set_title('Total Counts per Cell')
ax.legend()

# Gene expression correlation
ax = axes[0, 2]
mean_ref = counts_ref.mean(axis=0)
mean_easy = counts_easy.mean(axis=0)
mean_hard = counts_hard.mean(axis=0)
ax.scatter(np.log1p(mean_ref), np.log1p(mean_easy), alpha=0.7, label='Easy', color='green', s=30)
ax.scatter(np.log1p(mean_ref), np.log1p(mean_hard), alpha=0.7, label='Hard', color='red', s=30)
ax.plot([0, 4], [0, 4], 'k--', alpha=0.5)
ax.set_xlabel('Reference log(mean expr)')
ax.set_ylabel('Simulated log(mean expr)')
ax.set_title('Gene Expression Correlation')
ax.legend()

# Row 2: Cell type proportions
ax = axes[1, 0]
from collections import Counter
props_ref = np.array([Counter(fov_ref.class_instance).get(i, 0) for i in range(n_cell_types)]) / fov_ref.n_cells
props_easy = np.array([Counter(fov_easy.class_instance).get(i, 0) for i in range(n_cell_types)]) / fov_easy.n_cells
props_hard = np.array([Counter(fov_hard.class_instance).get(i, 0) for i in range(n_cell_types)]) / fov_hard.n_cells

x = np.arange(n_cell_types)
width = 0.25
ax.bar(x - width, props_ref, width, label='Reference', color='gray', alpha=0.7)
ax.bar(x, props_easy, width, label='Easy', color='green', alpha=0.7)
ax.bar(x + width, props_hard, width, label='Hard', color='red', alpha=0.7)
ax.set_xlabel('Cell Type')
ax.set_ylabel('Proportion')
ax.set_title('Cell Type Proportions')
ax.legend()

# Gene variance comparison
ax = axes[1, 1]
var_ref = counts_ref.var(axis=0)
var_easy = counts_easy.var(axis=0)
var_hard = counts_hard.var(axis=0)
ax.scatter(np.log1p(var_ref), np.log1p(var_easy), alpha=0.7, label='Easy', color='green', s=30)
ax.scatter(np.log1p(var_ref), np.log1p(var_hard), alpha=0.7, label='Hard', color='red', s=30)
ax.plot([0, 6], [0, 6], 'k--', alpha=0.5)
ax.set_xlabel('Reference log(variance)')
ax.set_ylabel('Simulated log(variance)')
ax.set_title('Gene Variance Correlation')
ax.legend()

# Summary metrics
ax = axes[1, 2]
metrics_to_plot = ['expression_similarity', 'correlation_similarity', 'type_proportion_similarity']
easy_vals = [metrics_easy.get(m, 0) for m in metrics_to_plot]
hard_vals = [metrics_hard.get(m, 0) for m in metrics_to_plot]

x = np.arange(len(metrics_to_plot))
width = 0.35
ax.bar(x - width/2, easy_vals, width, label='Easy', color='green', alpha=0.7)
ax.bar(x + width/2, hard_vals, width, label='Hard', color='red', alpha=0.7)
ax.set_ylabel('Similarity Score')
ax.set_title('Validation Metrics Summary')
ax.set_xticks(x)
ax.set_xticklabels(['Expression', 'Correlation', 'Type Props'], rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 1.1)

plt.suptitle('Validation: Comparing Simulated to Reference Data', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Comprehensive Dataset Evaluation

Evaluate a batch of simulations to understand variability in difficulty.

In [ ]:
# Generate multiple FOVs with varying parameters
from tqdm import tqdm

concentrations = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
n_replicates = 3

evaluation_results = []

for conc in tqdm(concentrations, desc="Evaluating concentrations"):
    for rep in range(n_replicates):
        # Create tissue with this concentration
        np.random.seed(int(conc * 1000) + rep)
        
        tissue_test = TissueCellTypes()
        tissue_test.generate_types_and_markers(
            n_genes=n_genes,
            n_cell_types=n_cell_types,
            expected_level=12.0,
            concentration=conc,
        )
        
        # Generate FOV
        fov_test, hybiss_test = generate_fov(tissue_test)
        dots_test = hybiss_test.make_pandas_df()
        counts_test = get_count_matrix(fov_test, dots_test, tissue_test)
        
        # Score difficulty
        scorer_test = DifficultyScorer(tissue_test)
        difficulty = scorer_test.score(fov_test)
        
        # Estimate classifier accuracy
        if counts_test.sum() > 0:
            try:
                acc, _ = estimate_classifier_accuracy(
                    counts_test, fov_test.class_instance, 'rf'
                )
            except:
                acc = 0.0
        else:
            acc = 0.0
        
        evaluation_results.append({
            'concentration': conc,
            'replicate': rep,
            'separability': difficulty.get('separability', 0),
            'classifier_accuracy': acc,
            'n_cells': fov_test.n_cells,
            'n_dots': len(dots_test),
        })

eval_df = pd.DataFrame(evaluation_results)
print(f"Evaluated {len(eval_df)} simulations")

In [ ]:
# Visualize the relationship between parameters and difficulty
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Concentration vs Separability
ax = axes[0]
for conc in eval_df['concentration'].unique():
    data = eval_df[eval_df['concentration'] == conc]
    ax.scatter([conc] * len(data), data['separability'], alpha=0.7, s=60)
means = eval_df.groupby('concentration')['separability'].mean()
ax.plot(means.index, means.values, 'r-', linewidth=2, marker='s', markersize=10)
ax.set_xlabel('Marker Concentration')
ax.set_ylabel('Separability Score')
ax.set_title('Concentration → Separability')

# Concentration vs Accuracy
ax = axes[1]
for conc in eval_df['concentration'].unique():
    data = eval_df[eval_df['concentration'] == conc]
    ax.scatter([conc] * len(data), data['classifier_accuracy'], alpha=0.7, s=60)
means = eval_df.groupby('concentration')['classifier_accuracy'].mean()
ax.plot(means.index, means.values, 'r-', linewidth=2, marker='s', markersize=10)
ax.set_xlabel('Marker Concentration')
ax.set_ylabel('Classifier Accuracy')
ax.set_title('Concentration → Classification Accuracy')
ax.set_ylim(0, 1.05)

# Separability vs Accuracy
ax = axes[2]
scatter = ax.scatter(eval_df['separability'], eval_df['classifier_accuracy'],
                     c=eval_df['concentration'], cmap='viridis', s=60, alpha=0.8)
plt.colorbar(scatter, ax=ax, label='Concentration')
ax.set_xlabel('Separability Score')
ax.set_ylabel('Classifier Accuracy')
ax.set_title('Separability ↔ Accuracy')

# Fit regression line
from scipy import stats
slope, intercept, r, p, se = stats.linregress(eval_df['separability'], eval_df['classifier_accuracy'])
x_line = np.linspace(eval_df['separability'].min(), eval_df['separability'].max(), 100)
ax.plot(x_line, slope * x_line + intercept, 'r--', label=f'r={r:.2f}')
ax.legend()

plt.suptitle('Parameter Sweep: Understanding Difficulty Drivers', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Creating Difficulty-Calibrated Datasets

Use difficulty scoring to create datasets with specific difficulty levels.

In [ ]:
def find_concentration_for_accuracy(target_accuracy, tolerance=0.05):
    """
    Binary search to find concentration parameter that achieves target accuracy.
    """
    low, high = 0.4, 0.99
    
    for _ in range(10):  # Max iterations
        mid = (low + high) / 2
        
        # Generate tissue with this concentration
        np.random.seed(42)
        tissue_test = TissueCellTypes()
        tissue_test.generate_types_and_markers(
            n_genes=n_genes,
            n_cell_types=n_cell_types,
            expected_level=12.0,
            concentration=mid,
        )
        
        fov_test, hybiss_test = generate_fov(tissue_test)
        dots_test = hybiss_test.make_pandas_df()
        counts_test = get_count_matrix(fov_test, dots_test, tissue_test)
        
        try:
            acc, _ = estimate_classifier_accuracy(counts_test, fov_test.class_instance, 'rf')
        except:
            acc = 0.5
        
        print(f"  Concentration {mid:.3f} → Accuracy {acc:.1%}")
        
        if abs(acc - target_accuracy) < tolerance:
            return mid, acc
        elif acc < target_accuracy:
            low = mid  # Need higher concentration
        else:
            high = mid  # Need lower concentration
    
    return mid, acc

# Find parameters for different difficulty levels
print("Finding parameters for 70% accuracy:")
conc_70, acc_70 = find_concentration_for_accuracy(0.70)

print(f"\nResult: concentration={conc_70:.3f} achieves {acc_70:.1%} accuracy")

---
## Summary

This notebook covered PointillSim's validation and difficulty assessment tools:

| Tool | Purpose |
|------|--------|
| `DifficultyScorer` | Quantify classification difficulty |
| `ValidationMetrics` | Compare simulated vs reference data |
| Classifier accuracy | Practical difficulty estimation |
| Parameter sweeps | Understanding difficulty drivers |

### Key Takeaways

1. **Separability** correlates strongly with classifier accuracy
2. **Marker concentration** is a primary driver of difficulty
3. **Validation metrics** help ensure simulations are realistic
4. **Calibration** allows creating datasets with target difficulty

### Next Steps

- **10_advanced_structures.ipynb**: Explore advanced histological elements
- **11_technology_presets.ipynb**: Platform-specific simulations